# 05_ann_lsh: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/05_ann_lsh/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Hash a product into buckets using random hyperplane signs.

Concepts: Hash a product into buckets using random hyperplane signs.
Run: uv run python modules/05_ann_lsh/examples/01_from_scratch.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

import numpy as np

from common.index_lessons import unit_catalog
from tinyvec.lsh import LSH


def main():
    products, matrix, query = unit_catalog(240)
    index = LSH(matrix, bits=6, tables=4, seed=17)
    # KEY IDEA: only rows sharing a complete signature in at least one table survive.
    candidates = index.candidates(query, 10)
    exact = np.argsort(-(matrix @ query))[:10]
    print(
        "rows inspected",
        len(candidates),
        "of",
        len(matrix),
        "true neighbors retained",
        len(set(exact) & set(candidates)),
    )
    print(products.iloc[candidates].title.head().tolist())
    assert len(candidates) == len(set(candidates))


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/05_ann_lsh/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Search binary codes with FAISS IndexLSH.

    Concepts: Search binary codes with FAISS IndexLSH.
    Run: uv run --extra libs python modules/05_ann_lsh/examples/02_production.py
    Expected: Inspect the printed IDs or measurements and the checked invariant.
    CPU: seconds on the included small workload. Uses local data; no Docker or model download.
    """

    # course-extra: libs
    import faiss

    from common.index_lessons import unit_catalog

    def main():
        products, matrix, q = unit_catalog(640)
        faiss.omp_set_num_threads(1)
        index = faiss.IndexLSH(matrix.shape[1], 64)
        index.add(matrix)
        distances, rows = index.search(q[None, :], 10)
        print(
            "Hamming distances",
            distances[0],
            "candidate products",
            products.iloc[rows[0]].id.tolist(),
        )
        assert all(distances[0][:-1] <= distances[0][1:])

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/05_ann_lsh/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure how hash table count changes coverage.

Concepts: Measure how hash table count changes coverage.
Run: uv run python modules/05_ann_lsh/examples/03_measure.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

from common.index_lessons import trial
from common.teaching import evidence


def main():
    observations = [trial("lsh", tables=t, bits=6) for t in (1, 4, 12)]
    print(observations)
    evidence("lesson-05_ann_lsh", {"observations": observations}, tier=2)


if __name__ == "__main__":
    main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.index_lessons import trial

print(trial("lsh", bits=6, tables=2))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.